# 20 — Fracture & MLS Triage Dependency / Oracle Audit

## Purpose

This notebook applies the same diagnostic logic previously used for ICH to the other two triage subproblems:

- skull fracture,
- midline shift (MLS).

It answers:

1. How much does perfect fracture prediction improve triage?
2. How much does perfect MLS prediction improve triage?
3. How much do fracture and MLS matter together?
4. How many ground-truth triage labels actually depend on fracture or MLS?
5. How many current predicted triage labels are being changed by each component?
6. Which fracture false positives / false negatives matter for triage?
7. Which MLS thresholds (1, 3, 5 mm) cause the most errors?
8. Does exact MLS regression matter, or is correct threshold-bin classification enough?
9. How are fracture boxes and MLS keypoints distributed across consecutive slices?
10. Do the annotations suggest that series context, expanded negatives, or more robust aggregation should be tested next?

Locked TEST is used only for descriptive milestone analysis. No threshold is tuned on TEST.

## 1. Environment

In [ ]:
!pip install -q pydicom

## 2. Imports

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, precision_score, recall_score

warnings.filterwarnings("ignore", message="Invalid value for VR UI.*")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 180)

## 3. Paths

In [ ]:
PREDICTION_ROOT = Path("/kaggle/input/datasets/mehdipaykanheyrati/final-evaluation/final_evaluation")

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/mehdipaykanheyrati/iaaa-contest-bct"),
    Path("/kaggle/input/iaaa-contest-bct"),
]

OUTPUT_ROOT = Path("/kaggle/working/fracture_mls_triage_audit")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEV_PATH = PREDICTION_ROOT / "common_dev_predictions.csv"
TEST_PATH = PREDICTION_ROOT / "common_test_predictions_LOCKED.csv"

if not DEV_PATH.exists():
    raise FileNotFoundError(DEV_PATH)
if not TEST_PATH.exists():
    raise FileNotFoundError(TEST_PATH)

print("DEV predictions:", DEV_PATH)
print("TEST predictions:", TEST_PATH)
print("Output:", OUTPUT_ROOT)

## 4. Official triage rule

In [ ]:
ICH_COLUMNS = ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]

def triage_from_values(values):
    V_EDH = max(0.0, float(values["V_EDH"]))
    V_SDH = max(0.0, float(values["V_SDH"]))
    V_IPH = max(0.0, float(values["V_IPH"]))
    V_SAH = max(0.0, float(values["V_SAH"]))
    V_IVH = max(0.0, float(values["V_IVH"]))
    fracture_prob = float(values["fracture_prob"])
    MLS_mm = max(0.0, float(values["MLS_mm"]))
    total_vol = V_EDH + V_SDH + V_IPH + V_SAH + V_IVH

    has_ich = total_vol >= 0.1
    fracture_present = fracture_prob >= 0.5

    if MLS_mm >= 5.0 and (has_ich or fracture_present): return 2
    if V_EDH >= 30.0: return 2
    if V_SDH >= 70.0: return 2
    if V_IPH >= 70.0: return 2
    if total_vol >= 60.0: return 2
    if has_ich and MLS_mm >= 3.0 and total_vol >= 40.0: return 2
    if fracture_present and total_vol >= 15.0: return 2
    if MLS_mm >= 5.0 and not (has_ich or fracture_present): return 1
    if has_ich: return 1
    if 3.0 <= MLS_mm < 5.0: return 1
    if fracture_present and total_vol < 15.0: return 1
    return 0

def macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="macro", labels=[0, 1, 2], zero_division=0))

def binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if tn + fp else np.nan,
        "FPR": float(fp / (tn + fp)) if tn + fp else np.nan,
    }

## 5. Load and normalize prediction tables

In [ ]:
def normalize_series_id(value):
    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()

def load_predictions(path):
    df = pd.read_csv(path).copy()
    df["series_id"] = df["series_id"].map(normalize_series_id)
    df["true_total_ich"] = df[[f"true_{column}" for column in ICH_COLUMNS]].sum(axis=1)
    df["pred_total_ich"] = df[[f"pred_{column}" for column in ICH_COLUMNS]].sum(axis=1)
    df["true_fracture"] = (df["true_fracture_prob"] >= 0.5).astype(int)
    df["pred_fracture"] = (df["pred_fracture_prob"] >= 0.5).astype(int)
    return df

datasets = {
    "DEV": load_predictions(DEV_PATH),
    "TEST_LOCKED": load_predictions(TEST_PATH),
}

for split, df in datasets.items():
    print(split, len(df), "series")

## 6. Single-component and pairwise oracle analysis

In [ ]:
def scenario_triage(df, use_gt_ich=False, use_gt_fracture=False, use_gt_mls=False):
    predictions = []

    for row in df.itertuples(index=False):
        values = {}
        for column in ICH_COLUMNS:
            values[column] = float(getattr(row, f"true_{column}" if use_gt_ich else f"pred_{column}"))
        values["fracture_prob"] = float(getattr(row, "true_fracture_prob" if use_gt_fracture else "pred_fracture_prob"))
        values["MLS_mm"] = float(getattr(row, "true_MLS_mm" if use_gt_mls else "pred_MLS_mm"))
        predictions.append(triage_from_values(values))

    return np.asarray(predictions, dtype=int)

oracle_rows = []

SCENARIOS = [
    ("Current", False, False, False),
    ("Perfect ICH only", True, False, False),
    ("Perfect fracture only", False, True, False),
    ("Perfect MLS only", False, False, True),
    ("Perfect fracture + MLS", False, True, True),
    ("Perfect ICH + fracture", True, True, False),
    ("Perfect ICH + MLS", True, False, True),
    ("All ground truth", True, True, True),
]

for split, df in datasets.items():
    baseline = macro_f1(df["true_triage"], df["pred_triage"])

    for name, gt_ich, gt_fracture, gt_mls in SCENARIOS:
        predictions = scenario_triage(df, gt_ich, gt_fracture, gt_mls)
        score = macro_f1(df["true_triage"], predictions)
        oracle_rows.append({
            "split": split,
            "scenario": name,
            "macro_F1": score,
            "gain_vs_current": score - baseline,
            "accuracy": float(accuracy_score(df["true_triage"], predictions)),
            "n_triage_errors": int((predictions != df["true_triage"].to_numpy()).sum()),
        })

oracle_df = pd.DataFrame(oracle_rows)
oracle_df.to_csv(OUTPUT_ROOT / "01_oracle_analysis.csv", index=False)
display(oracle_df)

## 7. Ground-truth dependency: how many triage labels need fracture or MLS?

In [ ]:
dependency_rows = []

for split, df in datasets.items():
    true_labels = df["true_triage"].to_numpy(dtype=int)

    fracture_removed = []
    mls_removed = []
    both_removed = []

    for row in df.itertuples(index=False):
        base = {column: float(getattr(row, f"true_{column}")) for column in ICH_COLUMNS}
        base["fracture_prob"] = float(row.true_fracture_prob)
        base["MLS_mm"] = float(row.true_MLS_mm)

        fracture_values = dict(base)
        fracture_values["fracture_prob"] = 0.0
        fracture_removed.append(triage_from_values(fracture_values))

        mls_values = dict(base)
        mls_values["MLS_mm"] = 0.0
        mls_removed.append(triage_from_values(mls_values))

        both_values = dict(base)
        both_values["fracture_prob"] = 0.0
        both_values["MLS_mm"] = 0.0
        both_removed.append(triage_from_values(both_values))

    fracture_removed = np.asarray(fracture_removed)
    mls_removed = np.asarray(mls_removed)
    both_removed = np.asarray(both_removed)

    dependency_rows.append({
        "split": split,
        "n_series": len(df),
        "GT_labels_changed_if_fracture_removed": int((fracture_removed != true_labels).sum()),
        "GT_labels_changed_if_MLS_removed": int((mls_removed != true_labels).sum()),
        "GT_labels_changed_if_both_removed": int((both_removed != true_labels).sum()),
        "fracture_dependency_rate": float((fracture_removed != true_labels).mean()),
        "MLS_dependency_rate": float((mls_removed != true_labels).mean()),
        "fracture_or_MLS_dependency_rate": float((both_removed != true_labels).mean()),
    })

dependency_df = pd.DataFrame(dependency_rows)
dependency_df.to_csv(OUTPUT_ROOT / "02_ground_truth_triage_dependency.csv", index=False)
display(dependency_df)

## 8. Predicted-decision dependency: what is currently driving model triage?

In [ ]:
pred_dependency_rows = []

for split, df in datasets.items():
    current_labels = df["pred_triage"].to_numpy(dtype=int)
    without_fracture = []
    without_mls = []

    for row in df.itertuples(index=False):
        base = {column: float(getattr(row, f"pred_{column}")) for column in ICH_COLUMNS}
        base["fracture_prob"] = float(row.pred_fracture_prob)
        base["MLS_mm"] = float(row.pred_MLS_mm)

        fracture_values = dict(base)
        fracture_values["fracture_prob"] = 0.0
        without_fracture.append(triage_from_values(fracture_values))

        mls_values = dict(base)
        mls_values["MLS_mm"] = 0.0
        without_mls.append(triage_from_values(mls_values))

    without_fracture = np.asarray(without_fracture)
    without_mls = np.asarray(without_mls)

    pred_dependency_rows.append({
        "split": split,
        "predicted_labels_changed_if_fracture_removed": int((without_fracture != current_labels).sum()),
        "predicted_labels_changed_if_MLS_removed": int((without_mls != current_labels).sum()),
        "predicted_fracture_dependency_rate": float((without_fracture != current_labels).mean()),
        "predicted_MLS_dependency_rate": float((without_mls != current_labels).mean()),
    })

pred_dependency_df = pd.DataFrame(pred_dependency_rows)
pred_dependency_df.to_csv(OUTPUT_ROOT / "03_predicted_triage_dependency.csv", index=False)
display(pred_dependency_df)

## 9. Fracture task metrics and triage impact

In [ ]:
fracture_rows = []
fracture_case_tables = {}

for split, df in datasets.items():
    metrics = binary_metrics(df["true_fracture"], df["pred_fracture"])
    perfect_fracture_triage = scenario_triage(df, False, True, False)
    current_correct = df["pred_triage"].to_numpy() == df["true_triage"].to_numpy()
    fracture_correct = perfect_fracture_triage == df["true_triage"].to_numpy()

    fracture_rows.append({
        "split": split,
        "positive_series": int(df["true_fracture"].sum()),
        "negative_series": int((1 - df["true_fracture"]).sum()),
        **metrics,
        "triage_errors_fixed_by_perfect_fracture": int((~current_correct & fracture_correct).sum()),
        "triage_cases_harmed_by_perfect_fracture": int((current_correct & ~fracture_correct).sum()),
    })

    cases = df[["series_id", "true_triage", "pred_triage", "true_fracture_prob", "pred_fracture_prob", "true_total_ich", "pred_total_ich", "true_MLS_mm", "pred_MLS_mm"]].copy()
    cases["fracture_error_type"] = np.select(
        [
            (df["true_fracture"] == 0) & (df["pred_fracture"] == 1),
            (df["true_fracture"] == 1) & (df["pred_fracture"] == 0),
        ],
        ["false_positive", "false_negative"],
        default="correct",
    )
    cases["triage_fixed_by_perfect_fracture"] = (~current_correct & fracture_correct).astype(int)
    fracture_case_tables[split] = cases
    cases.to_csv(OUTPUT_ROOT / f"04_{split}_fracture_cases.csv", index=False)

fracture_df = pd.DataFrame(fracture_rows)
fracture_df.to_csv(OUTPUT_ROOT / "04_fracture_summary.csv", index=False)
display(fracture_df)

## 10. DEV-only fracture threshold sweep

In [ ]:
dev = datasets["DEV"]
threshold_rows = []

for threshold in np.arange(0.05, 0.96, 0.01):
    fracture_pred = (dev["pred_fracture_prob"] >= threshold).astype(int)
    task_f1 = float(f1_score(dev["true_fracture"], fracture_pred, zero_division=0))
    triage_predictions = []

    for row, fracture_flag in zip(dev.itertuples(index=False), fracture_pred):
        values = {column: float(getattr(row, f"pred_{column}")) for column in ICH_COLUMNS}
        values["fracture_prob"] = float(fracture_flag)
        values["MLS_mm"] = float(row.pred_MLS_mm)
        triage_predictions.append(triage_from_values(values))

    threshold_rows.append({
        "threshold": float(threshold),
        "fracture_F1": task_f1,
        "triage_macro_F1": macro_f1(dev["true_triage"], triage_predictions),
    })

fracture_threshold_df = pd.DataFrame(threshold_rows)
fracture_threshold_df.to_csv(OUTPUT_ROOT / "05_DEV_fracture_threshold_sweep.csv", index=False)

print("Best DEV fracture-F1 threshold:")
display(fracture_threshold_df.sort_values("fracture_F1", ascending=False).head(5))
print("Best DEV triage threshold:")
display(fracture_threshold_df.sort_values("triage_macro_F1", ascending=False).head(5))

## 11. MLS continuous and threshold metrics

In [ ]:
mls_rows = []
mls_threshold_rows = []

for split, df in datasets.items():
    error = df["pred_MLS_mm"] - df["true_MLS_mm"]
    mls_rows.append({
        "split": split,
        "MAE_mm": float(mean_absolute_error(df["true_MLS_mm"], df["pred_MLS_mm"])),
        "bias_pred_minus_true_mm": float(error.mean()),
        "median_absolute_error_mm": float(error.abs().median()),
        "underestimation_rate": float((error < 0).mean()),
        "overestimation_rate": float((error > 0).mean()),
    })

    for threshold in [1.0, 3.0, 5.0]:
        metrics = binary_metrics((df["true_MLS_mm"] >= threshold).astype(int), (df["pred_MLS_mm"] >= threshold).astype(int))
        mls_threshold_rows.append({"split": split, "threshold_mm": threshold, **metrics})

mls_df = pd.DataFrame(mls_rows)
mls_threshold_df = pd.DataFrame(mls_threshold_rows)
mls_df.to_csv(OUTPUT_ROOT / "06_mls_continuous_metrics.csv", index=False)
mls_threshold_df.to_csv(OUTPUT_ROOT / "07_mls_threshold_metrics.csv", index=False)

display(mls_df)
display(mls_threshold_df)

## 12. Exact MLS oracle versus correct threshold-bin oracle

In [ ]:
def representative_mls(true_value):
    if true_value < 1.0:
        return 0.0
    if true_value < 3.0:
        return 1.5
    if true_value < 5.0:
        return 3.5
    return 5.0

mls_oracle_rows = []

for split, df in datasets.items():
    exact_predictions = scenario_triage(df, False, False, True)
    bin_predictions = []

    for row in df.itertuples(index=False):
        values = {column: float(getattr(row, f"pred_{column}")) for column in ICH_COLUMNS}
        values["fracture_prob"] = float(row.pred_fracture_prob)
        values["MLS_mm"] = representative_mls(float(row.true_MLS_mm))
        bin_predictions.append(triage_from_values(values))

    bin_predictions = np.asarray(bin_predictions, dtype=int)

    mls_oracle_rows.append({
        "split": split,
        "current_macro_F1": macro_f1(df["true_triage"], df["pred_triage"]),
        "exact_MLS_oracle_macro_F1": macro_f1(df["true_triage"], exact_predictions),
        "threshold_bin_oracle_macro_F1": macro_f1(df["true_triage"], bin_predictions),
        "exact_vs_bin_triage_disagreements": int((exact_predictions != bin_predictions).sum()),
    })

mls_oracle_df = pd.DataFrame(mls_oracle_rows)
mls_oracle_df.to_csv(OUTPUT_ROOT / "08_mls_exact_vs_bin_oracle.csv", index=False)
display(mls_oracle_df)

## 13. MLS threshold-crossing cases and triage impact

In [ ]:
mls_crossing_parts = []

for split, df in datasets.items():
    exact_mls_triage = scenario_triage(df, False, False, True)
    current_correct = df["pred_triage"].to_numpy() == df["true_triage"].to_numpy()
    mls_correct = exact_mls_triage == df["true_triage"].to_numpy()

    for threshold in [1.0, 3.0, 5.0]:
        true_flag = df["true_MLS_mm"] >= threshold
        pred_flag = df["pred_MLS_mm"] >= threshold
        crossed = true_flag != pred_flag
        temp = df.loc[crossed, ["series_id", "true_triage", "pred_triage", "true_MLS_mm", "pred_MLS_mm", "true_total_ich", "pred_total_ich", "true_fracture_prob", "pred_fracture_prob"]].copy()
        temp["split"] = split
        temp["threshold_mm"] = threshold
        temp["crossing_type"] = np.where(temp["pred_MLS_mm"] >= threshold, "false_positive_crossing", "false_negative_crossing")
        temp["triage_fixed_by_perfect_MLS"] = (~current_correct[crossed] & mls_correct[crossed]).astype(int)
        mls_crossing_parts.append(temp)

mls_crossings_df = pd.concat(mls_crossing_parts, ignore_index=True)
mls_crossings_df.to_csv(OUTPUT_ROOT / "09_mls_threshold_crossing_cases.csv", index=False)

display(mls_crossings_df.groupby(["split", "threshold_mm", "crossing_type"]).size().reset_index(name="count"))

## 14. Minimal component repair for each triage error

In [ ]:
repair_rows = []

for split, df in datasets.items():
    for row in df.itertuples(index=False):
        if int(row.pred_triage) == int(row.true_triage):
            continue

        fixes = []

        for name, gt_ich, gt_fracture, gt_mls in [
            ("ICH", True, False, False),
            ("fracture", False, True, False),
            ("MLS", False, False, True),
            ("fracture+MLS", False, True, True),
            ("ICH+fracture", True, True, False),
            ("ICH+MLS", True, False, True),
            ("ICH+fracture+MLS", True, True, True),
        ]:
            values = {}
            for column in ICH_COLUMNS:
                values[column] = float(getattr(row, f"true_{column}" if gt_ich else f"pred_{column}"))
            values["fracture_prob"] = float(getattr(row, "true_fracture_prob" if gt_fracture else "pred_fracture_prob"))
            values["MLS_mm"] = float(getattr(row, "true_MLS_mm" if gt_mls else "pred_MLS_mm"))

            if triage_from_values(values) == int(row.true_triage):
                fixes.append(name)

        smallest = min((fix.count("+") + 1 for fix in fixes), default=np.nan)
        minimal = [fix for fix in fixes if fix.count("+") + 1 == smallest] if fixes else []

        repair_rows.append({
            "split": split,
            "series_id": row.series_id,
            "true_triage": int(row.true_triage),
            "pred_triage": int(row.pred_triage),
            "minimal_fix_size": smallest,
            "minimal_fixes": " | ".join(minimal),
        })

repair_df = pd.DataFrame(repair_rows)
repair_df.to_csv(OUTPUT_ROOT / "10_minimal_component_repair.csv", index=False)
display(repair_df.groupby(["split", "minimal_fixes"]).size().reset_index(name="count").sort_values(["split", "count"], ascending=[True, False]))

## 15. Locate original annotations for structural context analysis

In [ ]:
def first_existing(paths):
    return next((path for path in paths if path.exists()), None)

DATASET_ROOT = first_existing(DATASET_ROOT_CANDIDATES)
if DATASET_ROOT is None:
    raise FileNotFoundError("CT dataset root was not found.")

DATA_ROOT = first_existing([DATASET_ROOT / "iaaa-contest-bct" / "Data", DATASET_ROOT / "Data"])
TRAINING_DIR = DATA_ROOT / "training"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"

TARGETS_PATH = first_existing([
    DATASET_ROOT / "series_targets_df.csv",
    DATASET_ROOT / "iaaa-contest-bct" / "series_targets_df.csv",
    DATA_ROOT / "series_targets_df.csv",
    DATA_ROOT.parent / "series_targets_df.csv",
])

targets_df = pd.read_csv(TARGETS_PATH).drop(columns=["Unnamed: 0"], errors="ignore").copy()
targets_df["series_id"] = targets_df["series_id"].map(normalize_series_id)

print("Training:", TRAINING_DIR)
print("Annotations:", ANNOTATIONS_DIR)

## 16. Build annotation-structure cache

In [ ]:
CACHE_PATH = OUTPUT_ROOT / "annotation_structure_cache.pkl"

def safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return float(default)

def point_to_line_distance_mm(points_xy, row_spacing, col_spacing):
    points = np.asarray(points_xy, dtype=np.float64)
    physical = np.column_stack([points[:, 0] * float(col_spacing), points[:, 1] * float(row_spacing)])
    anterior, posterior, outermost = physical
    reference = posterior - anterior
    length = float(np.linalg.norm(reference))
    if length < 1e-8:
        return 0.0
    relative = outermost - anterior
    cross = abs(reference[0] * relative[1] - reference[1] * relative[0])
    return float(cross / length)

if CACHE_PATH.exists():
    annotation_df = pd.read_pickle(CACHE_PATH)
else:
    rows = []

    for series_dir in sorted([path for path in TRAINING_DIR.iterdir() if path.is_dir()], key=lambda path: int(path.name)):
        series_id = normalize_series_id(series_dir.name)
        series_rows = []

        for dicom_path in series_dir.glob("*.dcm"):
            ds = pydicom.dcmread(dicom_path, stop_before_pixels=True, force=True)
            sop_uid = str(getattr(ds, "SOPInstanceUID", dicom_path.stem))
            position = getattr(ds, "ImagePositionPatient", None)
            orientation = getattr(ds, "ImageOrientationPatient", None)
            scalar = np.nan

            if position is not None and orientation is not None and len(position) >= 3 and len(orientation) >= 6:
                row_cosine = np.asarray(orientation[:3], dtype=float)
                col_cosine = np.asarray(orientation[3:6], dtype=float)
                scalar = float(np.dot(np.asarray(position[:3], dtype=float), np.cross(row_cosine, col_cosine)))

            spacing = getattr(ds, "PixelSpacing", [np.nan, np.nan])
            annotation_path = ANNOTATIONS_DIR / series_id / f"{sop_uid}.json"
            annotation_exists = int(annotation_path.exists())
            n_boxes = 0
            has_keypoints = 0
            slice_mls_mm = np.nan

            if annotation_path.exists():
                with open(annotation_path, "r", encoding="utf-8") as file:
                    annotation = json.load(file)

                n_boxes = len(annotation.get("boxes_xywh", []) or [])
                keypoints = annotation.get("keypoints", {}) or {}
                names = ["AnteriorFalxAttachment", "PosteriorFalxAttachment", "OutermostPointOfTheFalx"]

                if all(keypoints.get(name) is not None for name in names):
                    has_keypoints = 1
                    points = [keypoints[name] for name in names]
                    slice_mls_mm = point_to_line_distance_mm(points, float(spacing[0]), float(spacing[1]))

            series_rows.append({
                "series_id": series_id,
                "sop_uid": sop_uid,
                "position_scalar": scalar,
                "instance_number": safe_float(getattr(ds, "InstanceNumber", np.nan)),
                "annotation_exists": annotation_exists,
                "n_boxes": n_boxes,
                "has_keypoints": has_keypoints,
                "slice_mls_mm": slice_mls_mm,
            })

        temp = pd.DataFrame(series_rows)

        if temp["position_scalar"].notna().all():
            temp = temp.sort_values("position_scalar")
        elif temp["instance_number"].notna().all():
            temp = temp.sort_values("instance_number")
        else:
            temp = temp.sort_values("sop_uid")

        temp["slice_order"] = np.arange(len(temp))
        rows.append(temp)

    annotation_df = pd.concat(rows, ignore_index=True)
    annotation_df.to_pickle(CACHE_PATH)

print("Slices in annotation cache:", len(annotation_df))

## 17. Fracture ground-truth continuity and expanded-negative opportunity

In [ ]:
def run_lengths(flags):
    flags = np.asarray(flags, dtype=bool)
    runs = []
    start = None

    for index in range(len(flags) + 1):
        active = index < len(flags) and flags[index]
        if active and start is None:
            start = index
        if not active and start is not None:
            runs.append(index - start)
            start = None

    return runs

fracture_targets = targets_df[["series_id", "fracture_prob"]].copy()
fracture_targets["true_fracture"] = (fracture_targets["fracture_prob"] >= 0.5).astype(int)
structure = annotation_df.merge(fracture_targets, on="series_id", how="left")

fracture_runs = []
for series_id, group in structure[structure["true_fracture"] == 1].groupby("series_id"):
    group = group.sort_values("slice_order")
    fracture_runs.extend(run_lengths(group["n_boxes"] > 0))

fracture_structure_df = pd.DataFrame([{
    "fracture_positive_series": int(fracture_targets["true_fracture"].sum()),
    "box_positive_slices": int((structure["n_boxes"] > 0).sum()),
    "n_box_positive_runs": len(fracture_runs),
    "median_box_run_length": float(np.median(fracture_runs)) if fracture_runs else np.nan,
    "single_slice_box_run_rate": float(np.mean(np.asarray(fracture_runs) == 1)) if fracture_runs else np.nan,
    "run_le_2_rate": float(np.mean(np.asarray(fracture_runs) <= 2)) if fracture_runs else np.nan,
    "all_slices_in_true_fracture_negative_series": int((structure["true_fracture"] == 0).sum()),
    "annotated_slices_in_true_fracture_negative_series": int(((structure["true_fracture"] == 0) & (structure["annotation_exists"] == 1)).sum()),
}])

fracture_structure_df.to_csv(OUTPUT_ROOT / "11_fracture_annotation_structure.csv", index=False)
display(fracture_structure_df)

## 18. MLS annotation distribution and series-context structure

In [ ]:
mls_series = annotation_df.groupby("series_id").agg(
    n_slices=("slice_order", "size"),
    n_keypoint_slices=("has_keypoints", "sum"),
    max_annotated_slice_MLS_mm=("slice_mls_mm", "max"),
).reset_index()

mls_series = mls_series.merge(targets_df[["series_id", "MLS_mm"]], on="series_id", how="left")
positive_keypoint_counts = mls_series.loc[mls_series["n_keypoint_slices"] > 0, "n_keypoint_slices"]

keypoint_runs = []
for _, group in annotation_df.groupby("series_id"):
    group = group.sort_values("slice_order")
    keypoint_runs.extend(run_lengths(group["has_keypoints"] == 1))

mls_structure_df = pd.DataFrame([{
    "series_with_complete_keypoint_slices": int((mls_series["n_keypoint_slices"] > 0).sum()),
    "median_complete_keypoint_slices_per_positive_series": float(positive_keypoint_counts.median()),
    "mean_complete_keypoint_slices_per_positive_series": float(positive_keypoint_counts.mean()),
    "n_keypoint_runs": len(keypoint_runs),
    "median_keypoint_run_length": float(np.median(keypoint_runs)) if keypoint_runs else np.nan,
    "single_slice_keypoint_run_rate": float(np.mean(np.asarray(keypoint_runs) == 1)) if keypoint_runs else np.nan,
    "series_target_vs_max_annotated_slice_MAE_mm": float(mean_absolute_error(mls_series["MLS_mm"], mls_series["max_annotated_slice_MLS_mm"].fillna(0.0))),
}])

mls_structure_df.to_csv(OUTPUT_ROOT / "12_mls_annotation_structure.csv", index=False)
mls_series.to_csv(OUTPUT_ROOT / "13_mls_series_annotation_counts.csv", index=False)
display(mls_structure_df)

## 19. Direct answers

In [ ]:
answers = []

for split in ["DEV", "TEST_LOCKED"]:
    oracle = oracle_df[oracle_df["split"] == split].set_index("scenario")
    fracture = fracture_df[fracture_df["split"] == split].iloc[0]
    mls = mls_df[mls_df["split"] == split].iloc[0]
    dependency = dependency_df[dependency_df["split"] == split].iloc[0]
    mls_bin = mls_oracle_df[mls_oracle_df["split"] == split].iloc[0]

    answers.extend([
        {"split": split, "question": "Current triage performance", "answer": f"Macro F1 {oracle.loc['Current', 'macro_F1']:.4f}; accuracy {oracle.loc['Current', 'accuracy']:.4f}."},
        {"split": split, "question": "If fracture were perfect", "answer": f"Macro F1 {oracle.loc['Perfect fracture only', 'macro_F1']:.4f}; gain {oracle.loc['Perfect fracture only', 'gain_vs_current']:+.4f}."},
        {"split": split, "question": "If MLS were perfect", "answer": f"Macro F1 {oracle.loc['Perfect MLS only', 'macro_F1']:.4f}; gain {oracle.loc['Perfect MLS only', 'gain_vs_current']:+.4f}."},
        {"split": split, "question": "If fracture and MLS were both perfect", "answer": f"Macro F1 {oracle.loc['Perfect fracture + MLS', 'macro_F1']:.4f}; gain {oracle.loc['Perfect fracture + MLS', 'gain_vs_current']:+.4f}."},
        {"split": split, "question": "How much does true triage depend on fracture?", "answer": f"{int(dependency['GT_labels_changed_if_fracture_removed'])}/{int(dependency['n_series'])} labels ({100*dependency['fracture_dependency_rate']:.1f}%) change if true fracture is removed."},
        {"split": split, "question": "How much does true triage depend on MLS?", "answer": f"{int(dependency['GT_labels_changed_if_MLS_removed'])}/{int(dependency['n_series'])} labels ({100*dependency['MLS_dependency_rate']:.1f}%) change if true MLS is removed."},
        {"split": split, "question": "Current fracture task performance", "answer": f"F1 {fracture['F1']:.4f}; precision {fracture['precision']:.4f}; recall {fracture['recall']:.4f}; FP {int(fracture['FP'])}; FN {int(fracture['FN'])}."},
        {"split": split, "question": "Current MLS regression performance", "answer": f"MAE {mls['MAE_mm']:.3f} mm; bias {mls['bias_pred_minus_true_mm']:+.3f} mm."},
        {"split": split, "question": "Does exact MLS precision beyond threshold bins matter for triage?", "answer": f"Exact MLS oracle F1 {mls_bin['exact_MLS_oracle_macro_F1']:.4f}; correct-bin oracle F1 {mls_bin['threshold_bin_oracle_macro_F1']:.4f}; triage disagreements {int(mls_bin['exact_vs_bin_triage_disagreements'])}."},
    ])

answers.extend([
    {"split": "ANNOTATIONS", "question": "Are fractures often visible across neighboring slices?", "answer": f"Median GT box-positive run length is {fracture_structure_df['median_box_run_length'].iloc[0]:.1f} slices; single-slice run rate {100*fracture_structure_df['single_slice_box_run_rate'].iloc[0]:.1f}%."},
    {"split": "ANNOTATIONS", "question": "Is there a large expanded-negative opportunity for fracture?", "answer": f"True fracture-negative series contain {int(fracture_structure_df['all_slices_in_true_fracture_negative_series'].iloc[0])} total slices, compared with {int(fracture_structure_df['annotated_slices_in_true_fracture_negative_series'].iloc[0])} annotated slices."},
    {"split": "ANNOTATIONS", "question": "How many MLS-relevant slices usually exist?", "answer": f"Median complete-keypoint slices per positive series: {mls_structure_df['median_complete_keypoint_slices_per_positive_series'].iloc[0]:.1f}; median consecutive keypoint run: {mls_structure_df['median_keypoint_run_length'].iloc[0]:.1f}."},
])

answers_df = pd.DataFrame(answers)
answers_df.to_csv(OUTPUT_ROOT / "00_DIRECT_ANSWERS.csv", index=False)
display(answers_df)

# Questions this audit cannot answer without slice-level model outputs

The current evaluation CSV contains only one fracture probability and one MLS value per series. Therefore these questions need a later slice-level inference audit:

- Are fracture false-positive series caused by one isolated high-score slice because current aggregation uses a series maximum?
- Would requiring neighboring-slice support improve fracture precision without losing recall?
- Are MLS errors caused by one outlier slice because the current MLS aggregation takes the maximum shift among high-presence slices?
- Would median, trimmed maximum, percentile, or continuity-aware MLS aggregation outperform the current max rule?
- Would 2.5D context improve fracture detection or MLS keypoint regression?

This notebook identifies whether those experiments are worth prioritizing before building them.